# Match 4 AMD Validation Notebook

**Target:** AMD AI League - Match 4 (Intelligent Web Information Retrieval).

This notebook runs the agent through every required gate on the AMD
GPU node accessed via `notebooks.amd.com/hackathon`. Each section
is a self-contained verification step. Run them in order top-to-bottom.
The final cell prints a checklist of real statuses (NOT_RUN when a
gate has not been executed) you can paste into your submission notes.

> **Persistent directory rule** (from lablab FAQ):
> - `jupyter-hack-*` URLs  ->  save under `/persistent`
> - `rgapi-hackathon-*` URLs  ->  save under `/workspace`
> Anywhere else is wiped at session shutdown.


## 1. Environment check

In [ ]:
import os, socket, sys

print("Python:", sys.version)
# Detect host with graceful fallback (the `hostname` binary may be
# absent on minimal container images).
host = os.environ.get("JUPYTERHUB_HOSTNAME", "")
if not host:
    try:
        host = socket.gethostname()
    except Exception:
        host = "unknown"
print("Hostname:", host)
print("Working dir:", os.getcwd())
print("Env PATH[0:3]:", os.environ.get("PATH", "").split(":")[:3])
print("Detected host:", host)
if "jupyter-hack" in host:
    PERSIST = "/persistent"
elif "rgapi-hackathon" in host:
    PERSIST = "/workspace"
else:
    PERSIST = os.environ.get("MATCH4_PERSIST_DIR", "/persistent")
# If we can't write to the AMD persistent dir (e.g. in a slim
# validation image without those mount points), fall back to a
# tmpdir so the notebook continues to execute cleanly.
try:
    os.makedirs(PERSIST, exist_ok=True)
    test_path = os.path.join(PERSIST, ".match4-write-test")
    with open(test_path, "w") as f:
        f.write("ok")
    os.unlink(test_path)
except (PermissionError, OSError):
    import tempfile
    PERSIST = tempfile.mkdtemp(prefix="match4-persistent-")
    print(f"Falling back to writable persistent dir: {PERSIST}")
print(f"Persistent dir: {PERSIST}")
os.environ["MATCH4_PERSIST_DIR"] = PERSIST


## 2. Get the agent code

If this notebook was opened directly inside the repo, skip the clone.
Otherwise, set MATCH4_REPO_URL to your public submission URL.

In [ ]:
import os, subprocess

PERSIST = os.environ["MATCH4_PERSIST_DIR"]
REPO_DIR = os.path.join(PERSIST, "match4-amd-agent")
REPO_URL = os.environ.get("MATCH4_REPO_URL", "")

if not REPO_URL:
    # When run inside the slim validation image, the agent code is
    # already at /app (the workdir). When run on the AMD notebook
    # without a remote URL, we walk up from the current directory
    # until we find tests/ + app/app.py (the repo root marker).
    candidates = ["/app", os.getcwd()]
    cur = os.getcwd()
    while cur != os.path.dirname(cur):
        candidates.append(cur)
        cur = os.path.dirname(cur)
    for cand in candidates:
        if (os.path.isdir(os.path.join(cand, "tests"))
                and os.path.isfile(os.path.join(cand, "app", "app.py"))):
            REPO_DIR = cand
            break
    else:
        REPO_DIR = os.getcwd()
    print("Set MATCH4_REPO_URL to your public submission GitHub URL to clone.")
    print(f"Using local repo at: {REPO_DIR}")
elif not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])

print("Agent code at:", REPO_DIR)
os.environ["MATCH4_REPO_DIR"] = REPO_DIR


In [ ]:
import os, subprocess, sys

REPO_DIR = os.environ["MATCH4_REPO_DIR"]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "pytest", "pytest-asyncio",
     "aiohttp", "beautifulsoup4", "lxml"]
)
print("Deps installed")


## 3. Run unit + integration tests

In [ ]:
import os, subprocess, sys

REPO_DIR = os.environ["MATCH4_REPO_DIR"]
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=REPO_DIR, capture_output=True, text=True,
)
print("STDOUT (tail):", result.stdout[-2000:])
print("STDERR (tail):", result.stderr[-500:])
print("Return code:", result.returncode)
# Record the status. The notebook reports it; the gate check is in
# section 9. We deliberately do NOT raise on failure here — the
# notebook is a measurement tool, not a CI gate.
os.environ["MATCH4_TEST_RC"] = str(result.returncode)


## 4. CLI smoke test (offline, stub provider)

Confirms the CLI can run end-to-end without network access.

In [ ]:
import json, os, subprocess, sys, tempfile

REPO_DIR = os.environ["MATCH4_REPO_DIR"]
out_dir = tempfile.mkdtemp(prefix="match4-smoke-")
env = os.environ.copy()
env["MATCH4_SEARCH_PROVIDER"] = "stub"
env["MATCH4_OUTPUT_DIR"] = out_dir

r = subprocess.run(
    [sys.executable, "-m", "app.cli",
     "--query-id", "smoke_01",
     "--query", "How much does the TQ-40 cost?"],
    cwd=REPO_DIR, env=env, capture_output=True, text=True,
)
print("stdout:", r.stdout[:500])
print("stderr (tail):", r.stderr[-500:])
out_path = os.path.join(out_dir, "smoke_01_output.json")
print("output file exists:", os.path.isfile(out_path))
if os.path.isfile(out_path):
    with open(out_path) as f:
        data = json.load(f)
    print(json.dumps(data, indent=2))
    ok = all(k in data for k in ("answer", "citations", "confidence"))
    os.environ["MATCH4_SMOKE_OK"] = "1" if ok else "0"
else:
    os.environ["MATCH4_SMOKE_OK"] = "0"


## 5. Live test (DuckDuckGo, network-dependent)

This cell hits the public web. Gated on MATCH4_RUN_LIVE=1.

In [ ]:
import os, subprocess, sys, tempfile, json

if os.environ.get("MATCH4_RUN_LIVE") != "1":
    print("skipping live test - set MATCH4_RUN_LIVE=1 to enable")
else:
    REPO_DIR = os.environ["MATCH4_REPO_DIR"]
    out_dir = tempfile.mkdtemp(prefix="match4-live-")
    r = subprocess.run(
        [sys.executable, "-m", "app.cli",
         "--query-id", "live_01",
         "--query", "What is the price of the AMD Instinct MI300X?"],
        cwd=REPO_DIR, capture_output=True, text=True,
    )
    print("stderr (tail):", r.stderr[-800:])
    out_path = os.path.join(out_dir, "live_01_output.json")
    if os.path.isfile(out_path):
        with open(out_path) as f:
            data = json.load(f)
        print(json.dumps(data, indent=2))
    else:
        print("no output file produced; raw stdout:", r.stdout[:500])


## 6. Docker image build (optional)

Skip if Docker is not available on the notebook node. The build
may take several minutes; we set a 15-minute timeout.

In [ ]:
import os, subprocess

REPO_DIR = os.environ["MATCH4_REPO_DIR"]
docker_ok = subprocess.run(["which", "docker"], capture_output=True).returncode == 0
if not docker_ok:
    print("docker not available - skipping image build")
    os.environ["MATCH4_DOCKER_BUILD"] = "skipped:docker_unavailable"
else:
    # Per-call timeout so the cell never hangs on slow ROCm pulls.
    # On the AMD notebook the image is cached so this is fast.
    try:
        # Use the slim Dockerfile for the notebook (full ROCm pull is
        # not part of the default notebook verification — that lives
        # in the AMD-side manual workflow).
        r = subprocess.run(
            ["docker", "build", "-f", "Dockerfile.slim",
             "-t", "match4-amd-agent:test", "."],
            cwd=REPO_DIR, capture_output=True, text=True, timeout=300,
        )
        print("build stdout (tail):", r.stdout[-2000:])
        print("build stderr (tail):", r.stderr[-1000:])
        print("returncode:", r.returncode)
        os.environ["MATCH4_DOCKER_BUILD"] = (
            "ok" if r.returncode == 0 else "failed"
        )
    except subprocess.TimeoutExpired:
        print("docker build exceeded 300s - skipping")
        os.environ["MATCH4_DOCKER_BUILD"] = "timeout"


## 7. Docker grader-style test

Replicates the AMD grader pattern:
```
docker run ... python3 /app/app.py --query-id ... --query ...
docker exec ... cat /app/output/<query-id>_output.json
```

In [ ]:
import subprocess

if subprocess.run(["which", "docker"], capture_output=True).returncode != 0:
    print("docker not available - skipping")
else:
    r = subprocess.run(
        ["docker", "run", "--rm",
         "match4-amd-agent:test",
         "--query-id", "container_01",
         "--query", "What is the contact email for AMD support?"],
        capture_output=True, text=True, timeout=60,
    )
    print("stdout:", r.stdout[:500])
    print("stderr (tail):", r.stderr[-500:])
    print("returncode:", r.returncode)


## 8. ROCm torch visibility

Confirms the base image ships a ROCm-enabled torch build and can
see at least one GPU device. This is the canonical Match 3/4
verification step (no fabricated flags).

In [ ]:
import subprocess, sys

r = subprocess.run(
    [sys.executable, "-c",
     "import torch; "
     "print('torch', torch.__version__); "
     "print('rocm?', 'rocm' in torch.__version__); "
     "print('hip:', getattr(torch.version, 'hip', None)); "
     "print('cuda_api:', hasattr(torch, 'cuda')); "
     "print('visible:', torch.cuda.is_available() if hasattr(torch, 'cuda') else False); "
     "print('device:', torch.cuda.get_device_name(0) if (hasattr(torch, 'cuda') and torch.cuda.is_available()) else 'none')"],
    capture_output=True, text=True, timeout=30,
)
print(r.stdout)
print("stderr (tail):", r.stderr[-300:])


## 9. Verification summary

Paste the output below into your submission notes. Every status
is captured from real command output (NOT_RUN when the gate has
not been executed).

In [ ]:
import datetime, json, os, subprocess

# Capture per-section statuses from env vars set by previous cells.
def _safe_rc(name: str) -> str:
    try:
        return subprocess.run(
            ["bash", "-lc",
             f"echo ${{MATCH4_{name}:-NOT_RUN}}"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()
    except Exception:
        return "NOT_RUN"

checks = {
    "env_detected": "1" if os.environ.get("MATCH4_PERSIST_DIR") else "0",
    "repo_dir_set": "1" if os.environ.get("MATCH4_REPO_DIR") else "0",
    "tests_run": _safe_rc("TEST_RC"),
    "cli_smoke_offline": _safe_rc("SMOKE_OK"),
    "cli_live_network": "NOT_RUN (gated on MATCH4_RUN_LIVE=1)",
    "docker_build": _safe_rc("DOCKER_BUILD"),
    "docker_grader_style": _safe_rc("DOCKER_GRADER"),
    "rocm_torch": "NOT_RUN (only verifiable inside the ROCm base image)",
}

# Also fetch a tail of the pytest summary for the report.
try:
    bash_arg = 'cd "$MATCH4_REPO_DIR" && python -m pytest tests/ -q --tb=no 2>&1 | tail -3'
    test_summary = subprocess.run(
        ["bash", "-lc", bash_arg],
        capture_output=True, text=True, timeout=120,
    ).stdout.strip()
except Exception:
    test_summary = "(test summary unavailable)"

report = {
    "match": "AMD AI League Match 4 - Intelligent Web Information Retrieval",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "persistent_dir": os.environ.get("MATCH4_PERSIST_DIR"),
    "agent_repo": os.environ.get("MATCH4_REPO_DIR"),
    "test_summary_tail": test_summary,
    "checks": checks,
    "notes": [
        "AMD-side execution is REQUIRED for the official graded run.",
        "Do not hardcode answers to specific questions - the grader hides them.",
        "Container size, startup time, and per-query latency must each stay under the limits.",
        "Every check marked NOT_RUN must be replaced by its real status when executed on the AMD node.",
    ],
}
print(json.dumps(report, indent=2))
